In [ ]:
#!pip install seqeval
#!pip install -q "protobuf<5" "transformers>=4.35.0"

In [1]:
# libraries for data loading and system/path settings
import json
import sys
import warnings
import os
import random
import matplotlib.pyplot as plt
import numpy as np

# libraries for model building and training
from seqeval.metrics import classification_report as seqeval_classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from transformers import AutoTokenizer, AutoModelForTokenClassification, logging
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

# global settings to suppress unproblematic warning messages
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="The sentencepiece tokenizer")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" 

In [2]:
# load the augmented data in json format
with open("/kaggle/input/annotations-social-groups-augmentations/annotations_gpt_augmentations_corrected.json", "r") as f:
    data_augmented = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data_augmented:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

In [3]:
def tokenization_labelling(text, entities, tokenizer, tag2id, max_len):

    # get the encoding of the sentence
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True,
                         max_length=max_len, padding="max_length")
    
    # create preliminary list with O tags for all tokens
    tags = ["O"] * len(encoding.offset_mapping)

    # loop over annotations and extract start and end index as well as the given tag
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_tag = ent["tag"][0:2]

        # loop over all tokens in the sentence and check for overlap
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):
            # continue if it is a special token
            if token_start == token_end == 0:
                continue
            # check for overlap (overlap checking strictly necessary for deberta)
            if token_end > start and token_start < end:
                # assign B-tag if start is equal or smaller (smaller if deberta)
                if token_start <= start:
                    tags[idx] = f"B-{ent_tag}"
                # otherwise it is an inside token
                else:
                    tags[idx] = f"I-{ent_tag}"

    # extract the word ids
    word_ids = encoding.word_ids()

    # ensure propagate B-tags are propagated to all subwords of the same word (only actually relevant for deberta)
    for idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        # if token has B-tag ensure that all other tokens of the same word get I-tag
        if tags[idx].startswith("B-"):
            for j, wid2 in enumerate(word_ids):
                if wid2 == wid and j != idx:
                    tags[j] = f"I-{ent_tag}"

    # convert tags to IDs, masking special tokens
    tag_ids = [-100 if wid is None else tag2id.get(tag, tag2id["O"])
               for tag, wid in zip(tags, encoding.word_ids())]

    return encoding["input_ids"], encoding["attention_mask"], tag_ids, word_ids


class EarlyStopping:
    def __init__(self, patience, min_delta=0.0001, save_model=True, path='checkpoint.pt', printoption=False):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_f1 = None
        self.best_epoch = None
        self.early_stop = False
        self.path = path
        self.printoption = printoption
        self.save_model = save_model

    def __call__(self, current_f1, model, epoch):
        if self.best_f1 is None:
            self.best_f1 = current_f1
            self.best_epoch = epoch+1
            self.save_checkpoint(current_f1, model)
        elif current_f1 < self.best_f1 - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if current_f1 > self.best_f1:
                self.save_checkpoint(current_f1, model)
                self.best_f1 = current_f1
                self.best_epoch = epoch+1
                self.counter = 0

    def save_checkpoint(self, current_f1, model):
        if self.save_model:
            torch.save(model.state_dict(), self.path)
        if self.printoption:
            print(f'Validation F1 increased ({self.best_f1:.6f} --> {current_f1:.6f}).  Saving model ...')

def run_testset_ner(model, test_dataloader, id2tag, device, for_metric):
    
    model.eval()

    all_true_tags, all_pred_tags = [], []
    all_true_spans, all_pred_spans = [], []
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            tag_ids = batch["tag_ids"].to(device)
            batch_word_ids = batch["word_ids"]

            outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=tag_ids)
            logits = outputs.logits
            loss = outputs.loss
            total_loss += loss.item()
            num_batches += 1
            predictions = torch.argmax(logits, dim=2)

            if for_metric == "seqeval":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()

                    true_tags = []
                    pred_tags = []
                    for t, p in zip(true_seq, pred_seq):
                        if t != -100:
                            true_tags.append(id2tag[t])
                            pred_tags.append(id2tag[p])

                    all_true_tags.append(true_tags)
                    all_pred_tags.append(pred_tags)

            elif for_metric == "cross_span":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()
                    word_ids = batch_word_ids[i]

                    word_level_tags, _ = __labels_to_wordlevel_tags(true_seq, id2tag, word_ids)
                    all_true_spans.append(extract_spans(word_level_tags))

                    word_level_tags, _ = __labels_to_wordlevel_tags(pred_seq, id2tag, word_ids)
                    all_pred_spans.append(extract_spans(word_level_tags))

            elif for_metric == "sentence_level":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()
                    word_ids = batch_word_ids[i]

                    true_word_tags, _ = __labels_to_wordlevel_tags(true_seq, id2tag, word_ids)
                    pred_word_tags, _ = __labels_to_wordlevel_tags(pred_seq, id2tag, word_ids)

                    all_true_tags.append(true_word_tags)
                    all_pred_tags.append(pred_word_tags)

    avg_loss = total_loss / num_batches

    if for_metric == "seqeval":
        return all_true_tags, all_pred_tags, avg_loss
    elif for_metric == "cross_span":
        return all_true_spans, all_pred_spans, avg_loss
    elif for_metric == "sentence_level":
        return all_true_tags, all_pred_tags, avg_loss

def evaluate_seqeval(all_true_tags, all_pred_tags):
    classification_report = seqeval_classification_report(all_true_tags, all_pred_tags, output_dict=True)
    precision = classification_report["sg"]["precision"]
    recall = classification_report["sg"]["recall"]
    f1_score = classification_report["sg"]["f1-score"]

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1_score
        }

In [4]:
class TokenDataset(Dataset):
    def __init__(self, data, tokenizer, tag2id, max_len):
        self.dataset = []
        self.max_len = max_len

        for task in data:
            # get the sentence and all annotations
            text = task["sentence"]
            spans = task["annotations"]

            # tokenize and get all ids
            input_ids, attention_mask, tag_ids, word_ids = tokenization_labelling(text, spans, tokenizer, tag2id, self.max_len)

            # add everything to the dataset list
            self.dataset.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "tag_ids": torch.tensor(tag_ids, dtype=torch.long),
                "word_ids": word_ids})

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def custom_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    tag_ids = torch.stack([item["tag_ids"] for item in batch])
    word_ids = [item["word_ids"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "tag_ids": tag_ids,
        "word_ids": word_ids
    }

In [5]:
# set a seed to ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# set different dataset sizes to test
data_sizes = [3500, 4000, 4500, 5000]
num_folds = 5
f1_scores_non_augmented = []
f1_scores_augmented = []

# set model name and specific hyperparameters to test
model_name = "roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 9e-06
weight_decay = 0.01
batch_size = 16
epochs = 10
tokenizer = AutoTokenizer.from_pretrained(model_name)
augmentation_ratio = 0.25

# loop through different data sizes
for size in data_sizes:

    # take random subset according to the size
    data_subset = random.sample(data_augmented, k=size)

    print(f"\nCross-validation for dataset size = {size}")
    print("-" * 50)

    # create list to store fold metrics in
    fold_metrics_non_augmented = []
    fold_metrics_augmented = []

    # create K-Fold splits
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)

    for fold, (train_idx, val_idx) in enumerate(kf.split(data_subset)):
        
        # split the data
        train_fold_data = [data_subset[i] for i in train_idx]
        val_fold_data = [data_subset[i] for i in val_idx]

        # create augmented training data
        augmented_train_data = []
        num_augmentations_to_add = int(len(train_fold_data) * augmentation_ratio)
        candidates_for_augmentation = random.sample(
            train_fold_data, k=min(num_augmentations_to_add, len(train_fold_data))
        )
        for original_item in candidates_for_augmentation:
            if "augmentations" in original_item and len(original_item["augmentations"]) > 0:
                #aug = original_item["augmentations"][-1]
                aug = random.choice([
                    original_item["augmentations"][0],
                    original_item["augmentations"][-1]
                ])
                augmented_train_data.append({
                    "id": f"{original_item['id']}_aug_{aug['method']}",
                    "sentence": aug["sentence"],
                    "annotations": aug["annotations"]
                })
        train_fold_data_augmentations = train_fold_data + augmented_train_data

        # create datasets and dataloaders
        train_dataset = TokenDataset(train_fold_data, tokenizer, tag_to_id, max_len=128)
        train_dataset_augmentations = TokenDataset(train_fold_data_augmentations, tokenizer, tag_to_id, max_len=128)
        val_dataset = TokenDataset(val_fold_data, tokenizer, tag_to_id, max_len=128)
        
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
        train_augmentations_dataloader = DataLoader(train_dataset_augmentations, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

        # create model and optimizer for the non-augmentations
        model_non_augmented = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(tag_to_id),
            id2label=id_to_tag,
            label2id=tag_to_id
        ).to(device)
        optimizer_non_aug = AdamW(model_non_augmented.parameters(), lr=lr, weight_decay=weight_decay)

        # create early stopping object
        early_stopper_non_aug = EarlyStopping(patience=3, min_delta=0.0001, save_model=False, path='checkpoint.pt', printoption=False)

        # loop through the epochs
        for epoch in range(epochs):
            model_non_augmented.train()
            total_loss = 0.0
            for batch in train_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                tag_ids = batch["tag_ids"].to(device)

                optimizer_non_aug.zero_grad()

                with torch.autocast(device_type=device.type, dtype=torch.float16):
                    outputs = model_non_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                    loss = outputs.loss

                total_loss += loss.item()
                loss.backward()
                clip_grad_norm_(model_non_augmented.parameters(), 1.0)
                optimizer_non_aug.step()

            # calculate validation performance and feed into early stopper
            all_true, all_pred, _ = run_testset_ner(
                model=model_non_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
            )
            metrics_seqeval = evaluate_seqeval(all_true, all_pred)
            val_f1 = metrics_seqeval["f1"]
            early_stopper_non_aug(val_f1, model_non_augmented, epoch)
            if early_stopper_non_aug.early_stop:
                print("Early stopping triggered.")
                break    
                
        # get the best f1 score
        best_f1 = early_stopper_non_aug.best_f1
        fold_metrics_non_augmented.append(best_f1)
        print(f"F1 (non-augmentation): {best_f1}")

        # create model and optimizer for the augmentation model
        model_augmented = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(tag_to_id),
            id2label=id_to_tag,
            label2id=tag_to_id
        ).to(device)
        optimizer_aug = AdamW(model_augmented.parameters(), lr=lr, weight_decay=weight_decay)

        # create early stopping object
        early_stopper_aug = EarlyStopping(patience=3, min_delta=0.0001, save_model=False, path='checkpoint.pt', printoption=False)

        for epoch in range(epochs):
            model_augmented.train()
            total_loss = 0.0
            for batch in train_augmentations_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                tag_ids = batch["tag_ids"].to(device)

                optimizer_aug.zero_grad()

                with torch.autocast(device_type=device.type, dtype=torch.float16):
                    outputs = model_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                    loss = outputs.loss

                total_loss += loss.item()
                loss.backward()
                clip_grad_norm_(model_augmented.parameters(), 1.0)
                optimizer_aug.step()

            # calculate validation performance and feed into early stopper
            all_true, all_pred, _ = run_testset_ner(
                model=model_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
            )
            metrics_seqeval = evaluate_seqeval(all_true, all_pred)
            val_f1 = metrics_seqeval["f1"]
            early_stopper_aug(val_f1, model_augmented, epoch)
            if early_stopper_aug.early_stop:
                print("Early stopping triggered.")
                break    
                
        # get the best f1 score
        best_f1 = early_stopper_aug.best_f1
        fold_metrics_augmented.append(best_f1)
        print(f"F1 (augmentation): {best_f1}")

    # take averages across folds
    mean_seqeval_non_augmented = np.mean(fold_metrics_non_augmented)
    mean_seqeval_augmented = np.mean(fold_metrics_augmented)

    f1_scores_non_augmented.append(mean_seqeval_non_augmented)
    f1_scores_augmented.append(mean_seqeval_augmented)

    print(f"Mean F1 score (non-augmented): {mean_seqeval_non_augmented:.4f}")
    print(f"Mean F1 score (augmented): {mean_seqeval_augmented:.4f}")
    print("-" * 50)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


Cross-validation for dataset size = 3500
--------------------------------------------------


2025-11-17 16:15:13.749987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763396114.138360      89 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763396114.260495      89 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

F1 (non-augmentation): 0.7898989898989899
F1 (augmentation): 0.797558494404883
F1 (non-augmentation): 0.8194594594594595
F1 (augmentation): 0.8217391304347825
F1 (non-augmentation): 0.807606263982103
F1 (augmentation): 0.8053691275167785
F1 (non-augmentation): 0.8237585199610516
F1 (augmentation): 0.8175473579262214
F1 (non-augmentation): 0.828
Early stopping triggered.
F1 (augmentation): 0.8230694037145649
Mean F1 score (non-augmented): 0.8137
Mean F1 score (augmented): 0.8131
--------------------------------------------------

Cross-validation for dataset size = 4000
--------------------------------------------------
F1 (non-augmentation): 0.8014059753954306
F1 (augmentation): 0.8098159509202454
F1 (non-augmentation): 0.8055045871559634
F1 (augmentation): 0.8036529680365296
F1 (non-augmentation): 0.810344827586207
F1 (augmentation): 0.8144239226033423
Early stopping triggered.
F1 (non-augmentation): 0.8180924287118978
Early stopping triggered.
F1 (augmentation): 0.8255481410867495
Ea

In [5]:
# store the f1 scores
with open('/kaggle/working/f1_scores_non_augmented_3500_4000_4500_5000.json', 'w') as f:
    json.dump(f1_scores_non_augmented, f)

with open("/kaggle/working/f1_scores_augmented_3500_4000_4500_5000.json", "w") as f:
    json.dump(f1_scores_augmented, f)